In [9]:
from pathlib import Path

import fastf1
import pandas as pd

# Notebook runs from notebooks/, so root is one level up
PROJECT_ROOT = Path.cwd().parent
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE_DIR))

In [10]:
schedule = fastf1.get_event_schedule(2023, include_testing=False)
schedule[["RoundNumber", "EventName", "EventDate", "EventFormat"]]

,RoundNumber,EventName,EventDate,EventFormat
1,1,Bahrain Grand Prix,2023-03-05,conventional
2,2,Saudi Arabian Grand Prix,2023-03-19,conventional
3,3,Australian Grand Prix,2023-04-02,conventional
4,4,Azerbaijan Grand Prix,2023-04-30,sprint_shootout
5,5,Miami Grand Prix,2023-05-07,conventional
6,6,Monaco Grand Prix,2023-05-28,conventional
7,7,Spanish Grand Prix,2023-06-04,conventional
8,8,Canadian Grand Prix,2023-06-18,conventional
9,9,Austrian Grand Prix,2023-07-02,sprint_shootout
10,10,British Grand Prix,2023-07-09,conventional


In [11]:
def load_results(year, round_number, session_type):
    session = fastf1.get_session(year, round_number, session_type)
    session.load(laps=False, telemetry=False, weather=False, messages=False)
    return session.results

quali = load_results(2023, 1, "Q")
race = load_results(2023, 1, "R")

print(quali.columns.tolist())
print(race.columns.tolist())
race[["Abbreviation", "TeamName", "GridPosition", "Position", "Status"]]

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
co

['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']
['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']


,Abbreviation,TeamName,GridPosition,Position,Status
DriverNumber,,,,,
1,VER,Red Bull,1.0,1.0,Finished
11,PER,Red Bull,2.0,2.0,Finished
14,ALO,Aston Martin,5.0,3.0,Finished
55,SAI,Ferrari,4.0,4.0,Finished
44,HAM,Mercedes,7.0,5.0,Finished
18,STR,Aston Martin,8.0,6.0,Finished
63,RUS,Mercedes,6.0,7.0,Finished
77,BOT,Alfa Romeo,12.0,8.0,Finished
10,GAS,Alpine F1 Team,20.0,9.0,Finished


In [12]:
def best_quali_time(row):
    times = pd.to_timedelta(row[["Q1", "Q2", "Q3"]], errors="coerce")
    return times.min().total_seconds() if times.notna().any() else float("nan")

def collect_race(year, round_number, event_name, event_date):
    quali = load_results(year, round_number, "Q")
    race = load_results(year, round_number, "R")

    quali = quali[["Abbreviation", "Position", "Q1", "Q2", "Q3"]].copy()
    quali["BestQualiTime"] = quali.apply(best_quali_time, axis=1)
    quali = quali.rename(columns={"Position": "QualifyingPosition"})
    quali = quali[["Abbreviation", "QualifyingPosition", "BestQualiTime"]]

    race = race[["Abbreviation", "FullName", "TeamName", "GridPosition", "Position", "Status"]]
    race = race.rename(columns={"Position": "FinishingPosition"})

    data = race.merge(quali, on="Abbreviation", how="left", validate="one_to_one")

    # Grid 0 = pit lane start; put behind last grid slot
    pit_lane = data["GridPosition"] == 0
    data.loc[pit_lane, "GridPosition"] = data["GridPosition"].max() + 1
    data["PitLaneStart"] = pit_lane.astype(int)

    data["Podium"] = (data["FinishingPosition"] <= 3).astype(int)

    data.insert(0, "Year", year)
    data.insert(1, "Round", round_number)
    data.insert(2, "GrandPrix", event_name)
    data.insert(3, "EventDate", pd.to_datetime(event_date))
    return data

first = schedule.iloc[0]
bahrain = collect_race(2023, int(first["RoundNumber"]), first["EventName"], first["EventDate"])
bahrain

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '55', '14', '63', '44', '18', '31', '27', '4', '77', '24', '22', '23', '2', '20', '81', '21', '10']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
co

,Year,Round,GrandPrix,EventDate,Abbreviation,FullName,TeamName,GridPosition,FinishingPosition,Status,QualifyingPosition,BestQualiTime,PitLaneStart,Podium
0,2023,1,Bahrain Grand Prix,2023-03-05,VER,Max Verstappen,Red Bull,1.0,1.0,Finished,1.0,89.708,0,1
1,2023,1,Bahrain Grand Prix,2023-03-05,PER,Sergio Pérez,Red Bull,2.0,2.0,Finished,2.0,89.846,0,1
2,2023,1,Bahrain Grand Prix,2023-03-05,ALO,Fernando Alonso,Aston Martin,5.0,3.0,Finished,5.0,90.336,0,1
3,2023,1,Bahrain Grand Prix,2023-03-05,SAI,Carlos Sainz,Ferrari,4.0,4.0,Finished,4.0,90.154,0,0
4,2023,1,Bahrain Grand Prix,2023-03-05,HAM,Lewis Hamilton,Mercedes,7.0,5.0,Finished,7.0,90.384,0,0
5,2023,1,Bahrain Grand Prix,2023-03-05,STR,Lance Stroll,Aston Martin,8.0,6.0,Finished,8.0,90.836,0,0
6,2023,1,Bahrain Grand Prix,2023-03-05,RUS,George Russell,Mercedes,6.0,7.0,Finished,6.0,90.340,0,0
7,2023,1,Bahrain Grand Prix,2023-03-05,BOT,Valtteri Bottas,Alfa Romeo,12.0,8.0,Finished,12.0,91.443,0,0
8,2023,1,Bahrain Grand Prix,2023-03-05,GAS,Pierre Gasly,Alpine F1 Team,20.0,9.0,Finished,20.0,92.181,0,0
9,2023,1,Bahrain Grand Prix,2023-03-05,ALB,Alexander Albon,Williams,15.0,10.0,Finished,15.0,91.461,0,0


In [13]:
from fastf1.req import RateLimitExceededError

SEASON_DIR = RAW_DATA_DIR / "seasons"
SEASON_DIR.mkdir(parents=True, exist_ok=True)

def collect_season(year):
    path = SEASON_DIR / f"races_{year}.csv"
    if path.exists():
        print(f"{year}: loaded from disk")
        return pd.read_csv(path, parse_dates=["EventDate"])

    schedule = fastf1.get_event_schedule(year, include_testing=False)
    rows, failed = [], []
    for _, event in schedule.iterrows():
        try:
            rows.append(collect_race(year, event["RoundNumber"], event["EventName"], event["EventDate"]))
        except RateLimitExceededError:
            raise  # stop the run; re-run in an hour, cache keeps progress
        except Exception as error:
            failed.append((event["EventName"], str(error)))
    print(f"{year}: {len(rows)} ok, {len(failed)} failed {failed}")

    season = pd.concat(rows, ignore_index=True)
    if not failed:
        season.to_csv(path, index=False)  # only save full seasons
    return season

In [14]:
data = pd.concat([collect_season(y) for y in [2022, 2023, 2024, 2025]], ignore_index=True)
data = data.sort_values(["EventDate", "GridPosition"]).reset_index(drop=True)
data.shape

2022: loaded from disk
2023: loaded from disk
2024: loaded from disk
2025: loaded from disk


(1838, 14)

In [15]:
# Drop drivers who never started (withdrew after quali)
data = data[data["GridPosition"].notna()]
data = data[~((data["Year"] == 2023) & (data["Abbreviation"] == "STR") & (data["GrandPrix"] == "Singapore Grand Prix"))]
data = data.reset_index(drop=True)

podiums_per_race = data.groupby(["Year", "Round"])["Podium"].sum()
print("Races without exactly 3 podiums:\n", podiums_per_race[podiums_per_race != 3])
print(data.isna().sum())

data.to_csv(RAW_DATA_DIR / "races_2022_2025.csv", index=False)

Races without exactly 3 podiums:
 Series([], Name: Podium, dtype: int64)
Year                   0
Round                  0
GrandPrix              0
EventDate              0
Abbreviation           0
FullName               0
TeamName               0
GridPosition           0
FinishingPosition      0
Status                 0
QualifyingPosition     1
BestQualiTime         39
PitLaneStart           0
Podium                 0
dtype: int64
